# Phase 4: 하이브리드 검색 + RRF

Dense(Upstash) + BM25(Sparse)를 RRF로 합산한다.
Phase 1에서 발견한 문제(qa-001이 Java 검색에 0.89)가 해결되는지 확인.

In [1]:
from _bootstrap import setup_project_path

setup_project_path()

WindowsPath('C:/Users/mk.jang/Desktop/TLC/08_TSM/retrieval-lab')

In [2]:
from embedding_retrieval.config import RetrievalConfig
from embedding_retrieval.factory import create_embeddings
from embedding_retrieval.stores.dual_upstash import DualUpstashStore
from embedding_retrieval.search.bm25 import BM25Index
from embedding_retrieval.search.hybrid import HybridSearcher
from embedding_retrieval.scenarios.sample_data import SAMPLE_ENGINEER_PROFILES

config = RetrievalConfig.from_env()
embeddings = create_embeddings(config)
url = config.vector_store_kwargs["url"]
token = config.vector_store_kwargs["token"]

# 듀얼 스토어 (이미 02에서 업로드됨 — 재업로드해도 무방)
store = DualUpstashStore(embeddings=embeddings, url=url, token=token)

# BM25 인덱스 구축
cap_bm25 = BM25Index()
cap_bm25.fit([(p.engineer_id, p.capability_text) for p in SAMPLE_ENGINEER_PROFILES])

exp_bm25 = BM25Index()
exp_bm25.fit([(p.engineer_id, p.experience_text) for p in SAMPLE_ENGINEER_PROFILES])

# 하이브리드 검색기 생성
searcher = HybridSearcher(
    store=store,
    cap_bm25=cap_bm25,
    exp_bm25=exp_bm25,
    profiles=SAMPLE_ENGINEER_PROFILES,
)
print("HybridSearcher 준비 완료")

HybridSearcher 준비 완료


In [3]:
# 문제 케이스 재현: Dense에서 qa-001이 Java 검색에 0.89로 나왔던 것
# 하이브리드에서는 BM25가 qa-001을 걸러내므로 순위가 밀려야 함
print("=== 하이브리드: PL 포지션 (cap=0.2, exp=0.8) ===")
results = searcher.search(
    capability_query="Java / Spring",
    experience_query="제조업 ERP 시스템 개발. PL 역할. 팀장 역할 필수",
    weights=(0.2, 0.8),
    top_k=5,
    engineer_role="DEVELOPER",
    grades=["SENIOR", "INTERMEDIATE"],
)

print(f"{'rank':>4}  {'engineer':8}  {'final':>7}  {'cap_d':>6} {'cap_b':>6} {'exp_d':>6} {'exp_b':>6}")
print("-" * 65)
for c in results:
    s = c.score_breakdown
    print(f"{c.rank:4d}  {c.engineer_id:8s}  {s.final_score:.5f}  {s.capability_dense:.4f} {s.capability_bm25:.4f} {s.experience_dense:.4f} {s.experience_bm25:.4f}")

=== 하이브리드: PL 포지션 (cap=0.2, exp=0.8) ===
rank  engineer    final   cap_d  cap_b  exp_d  exp_b
-----------------------------------------------------------------
   1  eng-001   0.01634  0.9091 3.0422 0.8792 7.1960
   2  eng-002   0.01618  0.9576 3.1960 0.8544 1.9719
   3  eng-005   0.01429  0.8876 0.0000 0.8391 1.4562
   4  eng-003   0.01406  0.8629 0.0000 0.8342 0.8866


In [4]:
# 백엔드 개발자 (스킬 중심 cap=0.7, exp=0.3)
print("=== 하이브리드: 백엔드 개발자 (cap=0.7, exp=0.3) ===")
results = searcher.search(
    capability_query="Java / Spring",
    experience_query="현대차 ERP 시스템 개발. 백엔드 개발자. 현대차 프로젝트 경험 우대",
    weights=(0.7, 0.3),
    top_k=5,
    engineer_role="DEVELOPER",
    grades=["SENIOR", "INTERMEDIATE"],
)

print(f"{'rank':>4}  {'engineer':8}  {'final':>7}  {'cap_d':>6} {'cap_b':>6} {'exp_d':>6} {'exp_b':>6}")
print("-" * 65)
for c in results:
    s = c.score_breakdown
    print(f"{c.rank:4d}  {c.engineer_id:8s}  {s.final_score:.5f}  {s.capability_dense:.4f} {s.capability_bm25:.4f} {s.experience_dense:.4f} {s.experience_bm25:.4f}")

=== 하이브리드: 백엔드 개발자 (cap=0.7, exp=0.3) ===
rank  engineer    final   cap_d  cap_b  exp_d  exp_b
-----------------------------------------------------------------
   1  eng-002   0.01635  0.9576 3.1960 0.8917 8.9234
   2  eng-001   0.01617  0.9091 3.0422 0.9003 7.6840
   3  eng-005   0.01032  0.8876 0.0000 0.8626 4.4693
   4  eng-003   0.01016  0.8629 0.0000 0.8619 1.9033


In [5]:
# 프론트엔드 개발자 — React 정확 매칭이 중요한 케이스
print("=== 하이브리드: 프론트엔드 개발자 (cap=0.7, exp=0.3) ===")
results = searcher.search(
    capability_query="React / Chart.js",
    experience_query="대시보드 개발 경험. 프론트엔드 개발자. 차트.js 경험 우대",
    weights=(0.7, 0.3),
    top_k=5,
    engineer_role="DEVELOPER",
    only_full_time=False,
)

print(f"{'rank':>4}  {'engineer':8}  {'final':>7}  {'cap_d':>6} {'cap_b':>6} {'exp_d':>6} {'exp_b':>6}")
print("-" * 65)
for c in results:
    s = c.score_breakdown
    print(f"{c.rank:4d}  {c.engineer_id:8s}  {s.final_score:.5f}  {s.capability_dense:.4f} {s.capability_bm25:.4f} {s.experience_dense:.4f} {s.experience_bm25:.4f}")

=== 하이브리드: 프론트엔드 개발자 (cap=0.7, exp=0.3) ===
rank  engineer    final   cap_d  cap_b  exp_d  exp_b
-----------------------------------------------------------------
   1  eng-004   0.01639  0.9538 3.5090 0.9197 5.9964
   2  eng-003   0.01613  0.9162 1.4815 0.9121 5.3969
   3  eng-002   0.01028  0.8460 0.0000 0.8527 1.5075
   4  eng-005   0.01019  0.8433 0.0000 0.8529 1.4822
   5  eng-001   0.01000  0.8278 0.0000 0.8415 0.9911


In [10]:
# Dense only vs Hybrid 비교
print("=== Dense only: 'Java / Spring' capability ===")
dense_only = store.search(
    capability_query="Java / Spring",
    experience_query="제조업 ERP 시스템 개발",
    weights=(0.7, 0.3),
    top_k=5,
    engineer_role="DEVELOPER",
)
for r in dense_only:
    print(f"  {r.metadata['engineer_id']:8s}  score: {r.score:.4f}")

print()
print("=== Hybrid (Dense + BM25): 같은 쿼리 ===")
hybrid = searcher.search(
    capability_query="Java / Spring",
    experience_query="제조업 ERP 시스템 개발",
    weights=(0.7, 0.3),
    top_k=5,
    engineer_role="DEVELOPER",
)
for c in hybrid:
    print(f"  {c.engineer_id:8s}  score: {c.score_breakdown.final_score:.5f}")

print()
print("→ BM25가 키워드 매칭을 보강해서 Java/Spring 보유자가 더 확실하게 상위로 올라옴")

=== Dense only: 'Java / Spring' capability ===
  eng-002   score: 0.9094
  eng-001   score: 0.8819
  eng-005   score: 0.8539
  eng-003   score: 0.8343

=== Hybrid (Dense + BM25): 같은 쿼리 ===
  eng-002   score: 0.01631
  eng-001   score: 0.01621
  eng-005   score: 0.01032
  eng-003   score: 0.00781

→ BM25가 키워드 매칭을 보강해서 Java/Spring 보유자가 더 확실하게 상위로 올라옴
